# sabyinyo - model testing

Real evaluation of the checkpoints in `map-boy/sabyinyo-codegen`.

**Runtime > Change runtime type > T4 GPU** before running anything. On CPU the
perplexity cells take ~15 minutes instead of ~30 seconds.

**Colab secrets required** (key icon in the left sidebar, toggle "Notebook access"):

| Secret | Used for |
|---|---|
| `hug_read` | downloading checkpoints from the HF model repo |
| `KAGGLE_USERNAME` | downloading the corpus (tokenizer files + held-out text) |
| `KAGGLE_KEY` | same |
| `HF_TOKEN_WRITE` | only needed if you push new checkpoints; testing does not need it |

### What this notebook measures, and why

Generating a snippet and eyeballing it is not a test - a broken model and a
merely undertrained one both produce garbage. Every cell below compares the
model against a reference that makes the result interpretable:

1. **Held-out perplexity** on text the training run reserved and never saw,
   scored at the 2048-token length the model was trained on.
2. **Three baselines**: uniform-random, a unigram frequency table, and a
   randomly-initialised model of the same shape. A model that does not beat all
   three has learned nothing, no matter what its samples look like.
3. **Structural diagnostics** that isolate *which* component is at fault -
   architecture, initialisation, tokenizer, or training loop.
4. **A loss-vs-step curve** across checkpoints, which tells you whether more
   training would help or whether the run is stuck.


## 1. Setup

In [ ]:
import os, subprocess, sys, json, time

REPO_DIR = "/content/sabyinyo"
DATA_DIR = "/content/data"
BRANCH   = "claude/model-testing-colab-setup-634pwb"   # set to "main" once merged

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH,
                    "https://github.com/map-boy/sabyinyo.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub", "tokenizers", "kaggle"], check=True)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Everything still runs, just slowly.")

In [ ]:
from google.colab import userdata

def secret(*names):
    """Return the first Colab secret that exists, so the notebook is not
    hostage to one exact secret name."""
    for n in names:
        try:
            v = userdata.get(n)
            if v:
                return v
        except Exception:
            pass
    return None

HF_TOKEN = secret("hug_read", "HF_TOKEN_READ", "HF_TOKEN", "HF_TOKEN_WRITE")
KAGGLE_USERNAME = secret("KAGGLE_USERNAME")
KAGGLE_KEY = secret("KAGGLE_KEY")

assert HF_TOKEN, "No HF read token found. Add 'hug_read' in the Colab secrets panel."
assert KAGGLE_USERNAME and KAGGLE_KEY, "Add KAGGLE_USERNAME and KAGGLE_KEY as Colab secrets."

# Downstream libs read these from the environment.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("secrets loaded (values not printed)")

In [ ]:
# The corpus gives us two things testing needs: the tokenizer files, and the
# held-out tail of train.txt that the training run never trained on.
os.makedirs(DATA_DIR, exist_ok=True)
needed = ["vocab.json", "merges.txt", "train.txt"]

if not all(os.path.exists(f"{DATA_DIR}/{f}") for f in needed):
    subprocess.run(["kaggle", "datasets", "download",
                    "-d", "mugishaalainpaisible/codegen-corpus-v1",
                    "-p", DATA_DIR, "--unzip"], check=True)

for f in needed:
    path = f"{DATA_DIR}/{f}"
    if os.path.exists(path):
        print(f"  OK       {f}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"  MISSING  {f}")

## 2. One-command test run

`eval/run_eval.py` runs the whole suite and exits non-zero if anything fails, so
the same command works in CI. The rest of the notebook unpacks each piece
interactively - run this first to see the overall picture.

In [ ]:
!cd /content/sabyinyo && PYTHONPATH=/content/sabyinyo python eval/run_eval.py \
    --data-dir /content/data \
    --checkpoint latest \
    --compare-checkpoint step_100000.pt \
    --windows 8

## 3. What is actually on the Hub

In [ ]:
from eval.harness import list_checkpoints, download_checkpoint, HF_REPO_ID

files = list_checkpoints(HF_TOKEN)
steps = sorted(int(f.split("step_")[-1].split(".pt")[0]) for f in files if "step_" in f)
print(f"{len(files)} checkpoint files, steps {steps[0]} .. {steps[-1]}")
print("latest.pt present:", "checkpoints/latest.pt" in files)

from huggingface_hub import HfApi
for c in HfApi(token=HF_TOKEN).list_repo_commits(HF_REPO_ID, repo_type="model")[:5]:
    print(f"  {c.created_at:%Y-%m-%d %H:%M}  {c.title}")

## 4. Load the tokenizer - and check it before trusting it

`ByteLevelBPETokenizer(vocab.json, merges.txt)` loaded raw does **not** treat
`<|python|>` as one token. The ByteLevel pre-tokenizer splits it into
`<`, `|`, `python`, `|`, `>`. `token_to_id("<|python|>")` still returns `9`,
which makes the bug easy to miss - you have to inspect `encode()`.

`eval.harness.load_tokenizer` calls `add_special_tokens()` to fix this. The cell
below shows both, so you can see the difference.

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from eval.harness import load_tokenizer, SPECIAL_TOKENS
from eval import diagnostics as dx

raw = ByteLevelBPETokenizer(f"{DATA_DIR}/vocab.json", f"{DATA_DIR}/merges.txt")
tok = load_tokenizer(DATA_DIR)          # same files, special tokens registered

print(f"{'token':<16} {'token_to_id':>12} {'raw encode':>22} {'fixed encode':>16}")
for t in SPECIAL_TOKENS:
    print(f"{t:<16} {str(raw.token_to_id(t)):>12} "
          f"{str(raw.encode(t).ids):>22} {str(tok.encode(t).ids):>16}")

for r in (dx.test_tokenizer_roundtrip(tok), dx.test_special_tokens(tok)):
    print(f"\n{'PASS' if r['passed'] else 'FAIL'}  {r['name']}\n      {r['detail']}")

## 5. Load the checkpoint

`load_model` uses `strict=False` and then reports any mismatch itself, so if
`model/architecture.py` has drifted from what produced the checkpoint you get a
clear error naming the keys instead of silently evaluating half-random weights.

In [ ]:
from eval.harness import load_model

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_path = download_checkpoint("checkpoints/latest.pt", HF_TOKEN)
model, meta = load_model(ckpt_path, device)

print(f"step:     {meta['step']}")
print(f"params:   {meta['n_params']/1e6:.1f}M")
print(f"device:   {device}")
print(f"optimizer state in checkpoint: {meta['has_optimizer']}")

r = dx.test_weight_health(model)
print(f"\n{'PASS' if r['passed'] else 'FAIL'}  {r['name']}: {r['detail']}")

## 6. Held-out perplexity against three baselines

This is the number that matters. Notes on methodology, because each of these
choices changes the result by orders of magnitude:

- **Held-out, not training data.** `train.py` reserves the last 1% of the token
  stream for validation. We score the last 0.5% of *bytes*, which sits safely
  inside that region. Scoring `train.txt` from the front measures memorisation,
  not generalisation.
- **2048-token windows**, matching the training sequence length. Scoring
  512-token windows measures a regime the model was never trained in.
- **Two numbers reported.** `ppl_all` counts every token including the first few
  in each window, which have no context and are unpredictable by construction.
  `ppl_warm` counts only tokens with >= 256 tokens of context. `ppl_warm` is the
  honest number; a large gap between them just means your windows are short.
- **Baselines.** Perplexity in isolation means nothing. Uniform-random is the
  floor no LM may sit below. The unigram table is the floor any model that uses
  context must clear. The untrained model of identical shape tells you whether
  training accomplished anything at all.

In [ ]:
from eval.harness import (read_holdout_text, perplexity, unigram_baseline,
                          uniform_baseline, build_model)

holdout_text = read_holdout_text(f"{DATA_DIR}/train.txt")
holdout_ids = tok.encode(holdout_text).ids
print(f"held-out tokens: {len(holdout_ids):,}")
print(f"first 200 chars: {holdout_text[:200]!r}\n")

t0 = time.time()
ppl = perplexity(model, holdout_ids, device=device, seq_len=2048,
                 max_windows=8, verbose=True)
print(f"\nscored in {time.time()-t0:.1f}s")

uni  = unigram_baseline(holdout_ids)
unif = uniform_baseline()

torch.manual_seed(0)
untrained = build_model(device).eval()
untr = perplexity(untrained, holdout_ids, device=device, seq_len=2048, max_windows=2)
del untrained
if device == "cuda":
    torch.cuda.empty_cache()

print(f"\n{'':<24}{'loss':>10}{'perplexity':>16}")
print("-" * 50)
print(f"{'MODEL (held-out)':<24}{ppl['loss_warm']:>10.3f}{ppl['ppl_warm']:>16.1f}")
print(f"{'  ...all tokens':<24}{ppl['loss_all']:>10.3f}{ppl['ppl_all']:>16.1f}")
print(f"{'UNTRAINED same shape':<24}{untr['loss_warm']:>10.3f}{untr['ppl_warm']:>16.1f}")
print(f"{'UNIGRAM table':<24}{uni['loss']:>10.3f}{uni['ppl']:>16.1f}")
print(f"{'UNIFORM random':<24}{unif['loss']:>10.3f}{unif['ppl']:>16.1f}")

print()
for r in (dx.test_beats_uniform(ppl),
          dx.test_beats_unigram(ppl, uni),
          dx.test_beats_untrained(model, untr, ppl)):
    print(f"{'PASS' if r['passed'] else 'FAIL'}  {r['name']}\n      {r['detail']}\n")

## 7. Structural diagnostics

Perplexity says *how bad*. These say *why*. They run in seconds and none of them
needs a GPU.

In [ ]:
checks = [
    dx.test_positional_encoding_exists(),
    dx.test_init_scale(),
    dx.test_training_budget(),
    dx.test_order_sensitivity_on_real_model(model, holdout_ids, device=device),
    dx.test_output_entropy(model, holdout_ids, device=device),
]
for r in checks:
    print(f"{'PASS' if r['passed'] else 'FAIL'}  {r['name']}")
    print(f"      {r['detail']}\n")

### The permutation test, spelled out

`test_positional_encoding_exists` builds a **1-layer** instance of this exact
architecture and feeds it two prefixes that are permutations of each other,
ending on the same token. With causal attention and no positional signal, the
last position attends over the same *set* of key/value vectors either way, so
the logits come out bit-identical.

One layer is used deliberately: with 12 layers the outputs do differ slightly,
because each earlier position's representation depends on its own causal prefix
*set*, and those sets differ. That leak is not positional encoding - it is a
faint side channel. The 1-layer test removes the ambiguity.

Run it below on the real trained weights as well.

In [ ]:
# Same experiment, on the actual 12-layer trained model.
import torch.nn.functional as F

base = holdout_ids[512:640]
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(base) - 1, generator=g).tolist()
shuffled = [base[j] for j in perm] + [base[-1]]

with torch.no_grad():
    def probs(ids):
        lg = model(torch.tensor([ids], device=device))[0, -1].float()
        return torch.softmax(lg, dim=-1)
    p_orig, p_shuf = probs(base), probs(shuffled)

top_orig = torch.topk(p_orig, 5)
top_shuf = torch.topk(p_shuf, 5)
print("top-5 next tokens, ORIGINAL order:")
for pr, i in zip(top_orig.values.tolist(), top_orig.indices.tolist()):
    print(f"   {pr:6.3f}  {tok.decode([i])!r}")
print("top-5 next tokens, SHUFFLED prefix (same tokens, same last token):")
for pr, i in zip(top_shuf.values.tolist(), top_shuf.indices.tolist()):
    print(f"   {pr:6.3f}  {tok.decode([i])!r}")
print(f"\ntotal variation distance: {0.5*(p_orig-p_shuf).abs().sum().item():.4f}")
print("A healthy code model would be near 1.0 here: reordering a line of code "
      "completely changes what comes next.")

## 8. Is it still learning? Loss vs training step

Downloads several checkpoints and scores each on the same held-out windows. The
*shape* of this curve is the decision:

- still falling steeply -> the run is undertrained, keep going;
- flat and above the uniform baseline -> something is structurally broken and
  more steps will not fix it.

Each checkpoint is ~1.5 GB, so the cache is cleared between downloads to avoid
filling the Colab disk. Set `STEPS_TO_TEST` to fewer entries if you are short on
time.

In [ ]:
import shutil

STEPS_TO_TEST = [2000, 20000, 50000, 80000, 100000]
EVAL_WINDOWS  = 3            # same windows for every checkpoint -> comparable


def purge_hub_cache(path):
    """Delete the whole cached repo, blobs included.

    Deleting only the snapshot symlink leaves the 1.5 GB blob behind and fills
    the Colab disk after three or four checkpoints.
    """
    parts = path.split(os.sep)
    for j, part in enumerate(parts):
        if part.startswith("models--"):
            shutil.rmtree(os.sep.join(parts[:j + 1]), ignore_errors=True)
            return


curve = []
for step in STEPS_TO_TEST:
    try:
        path = download_checkpoint(f"checkpoints/step_{step}.pt", HF_TOKEN)
        m, mt = load_model(path, device)
        res = perplexity(m, holdout_ids, device=device, seq_len=2048,
                         max_windows=EVAL_WINDOWS)
        curve.append((step, res["loss_warm"], res["ppl_warm"]))
        print(f"step {step:>7}: loss={res['loss_warm']:7.3f}  ppl={res['ppl_warm']:.3g}")
        del m
        if device == "cuda":
            torch.cuda.empty_cache()
        purge_hub_cache(path)
    except Exception as e:
        print(f"step {step}: skipped - {type(e).__name__}: {e}")

print(f"\nuniform baseline loss = {unif['loss']:.3f}")
print(f"unigram baseline loss = {uni['loss']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

if curve:
    xs = [c[0] for c in curve]; ys = [c[1] for c in curve]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(xs, ys, "o-", label="held-out loss")
    ax.axhline(unif["loss"], ls="--", color="tab:red", label="uniform random")
    ax.axhline(uni["loss"], ls=":", color="tab:orange", label="unigram table")
    ax.set_xlabel("training step (micro-batches)")
    ax.set_ylabel("held-out cross-entropy (nats)")
    ax.set_title("sabyinyo-codegen: loss vs step")
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

### Did the last few checkpoints train anything?

`step_100000.pt` through `step_100005.pt` were written by separate resumed runs.
`train.py` counts **micro-batches** but only calls `optimizer.step()` every
`GRAD_ACCUM = 64` of them. A resumed run that starts at step 100000 hits
`step >= MAX_STEPS` after a handful of micro-batches - long before the
accumulation window closes - so no optimizer step ever happens and the "new"
checkpoint is a byte-for-byte copy of the old weights.

In [ ]:
a = download_checkpoint("checkpoints/step_100000.pt", HF_TOKEN)
b = download_checkpoint("checkpoints/latest.pt", HF_TOKEN)
r = dx.test_checkpoints_are_distinct(a, b, "step_100000", "latest")
print(f"{'PASS' if r['passed'] else 'FAIL'}  {r['name']}\n      {r['detail']}")

## 9. Generation

Two things the original testing got wrong, both fixed here:

- **Prompt format.** The corpus tags files as
  `<filename>path</filename>` then `<language>python</language>`. Prompting with
  `<|python|>` asks for a format that appears in the vocabulary but never in the
  training data - that is an out-of-distribution test, not a fair one.
- **Decoding.** Pure temperature sampling with no top-k/top-p produces noise from
  the tail, and pure greedy decoding on a weak model loops forever. Both are
  shown below alongside the sane default.

`CodeGenModel` has no KV cache, so each new token re-runs the full forward pass
over the whole prefix. Cost is quadratic - keep `max_new_tokens` modest.

In [ ]:
from eval.harness import generate, timed_generate

PROMPTS = [
    "<filename>utils/math_helpers.py</filename>\n<language>python</language>\n"
    "def fibonacci(n):\n",
    "<filename>src/types.ts</filename>\n<language>typescript</language>\n"
    "interface User {\n  id: number;\n",
    "<filename>scripts/backup.sh</filename>\n<language>bash</language>\n"
    "#!/usr/bin/env bash\nset -euo pipefail\n",
]

for prompt in PROMPTS:
    print("=" * 70)
    print(prompt)
    g = timed_generate(model, tok, prompt, max_new_tokens=48,
                       temperature=0.0, device=device)
    print(f"--- greedy ({g['tokens_per_second']:.1f} tok/s) ---")
    print(g["completion"])
    s = generate(model, tok, prompt, max_new_tokens=48, temperature=0.8,
                 top_k=50, top_p=0.95, repetition_penalty=1.1,
                 device=device, seed=0)
    print("--- sampled (T=0.8, top-k=50, top-p=0.95, rep-penalty=1.1) ---")
    print(s["completion"])
    print()

In [ ]:
# Free-form playground.
prompt = "<filename>app/main.py</filename>\n<language>python</language>\nimport os\n"

out = generate(model, tok, prompt,
               max_new_tokens=64,
               temperature=0.7,      # 0.0 = greedy
               top_k=50,
               top_p=0.95,
               repetition_penalty=1.15,
               device=device,
               seed=None)
print(out["full"])

## 10. Reading the results

**If the model beats all three baselines** and the loss curve is still falling:
it works, it is undertrained, add steps.

**If held-out loss sits above `ln(32000) = 10.37`**, the model is worse than
guessing uniformly at random. That is not undertraining - training never
converged, and no number of extra steps fixes it. The diagnostics in section 7
name the cause. `docs/FINDINGS.md` in this repo has the full analysis and the
specific patches.

The checks are ordered by how much they cost to fix, cheapest first: tokenizer
handling and prompt format are notebook-level fixes; the training-loop step
accounting is a few lines in `kaggle_kernel/train.py`; initialisation is a small
addition to `model/architecture.py`; a missing positional encoding means
retraining from scratch, because it changes what the weights mean.